# Feature Store - Despachos

## Membros do Grupo

- **Aloenia Oliveira**
- **Douglas Morales Monteiro**
- **Osnil Arruda**
- **Mickael Alves**
- **Pablo Cantu**
- **Thiago Sandoval**



In [0]:
WITH tb_pedidos AS (
  SELECT *
  FROM workspace.olist.orders o
  INNER JOIN workspace.olist.order_items oi on o.order_id = oi.order_id
  WHERE order_purchase_timestamp <= '2018-07-01'
)

SELECT seller_id,

    -- Percentual do valor do frete em relação ao produto - Vida
    coalesce(sum(freight_value) / sum(price) * 100, 0) as pct_frete_ped_vida,

    -- Percentual do valor do frete em relação ao produto - 28 dias
    coalesce(sum(case when order_purchase_timestamp >= date('2018-07-01') - interval '28 days' then freight_value end) / sum(case when order_purchase_timestamp >= date('2018-07-01') - interval '28 days' then price end) * 100, 0) as pct_frete_ped_28d,

    -- Percentual do valor do frete em relação ao produto - 56 dias
    coalesce(sum(case when order_purchase_timestamp >= date('2018-07-01') - interval '56 days' then freight_value end) / sum(case when order_purchase_timestamp >= date('2018-07-01') - interval '56 days' then price end), 0) * 100 as pct_frete_ped_56d,

    -- Percentual do valor do frete em relação ao produto - 365 dias
    coalesce(sum(case when order_purchase_timestamp >= date('2018-07-01') - interval '365 days' then freight_value end) / sum(case when order_purchase_timestamp >= date('2018-07-01') - interval '365 days' then price end), 0) * 100 as pct_frete_ped_365d,

    -- Média do valor do frete - Vida
    coalesce(avg(freight_value), 0) as avg_frete_vida,

    -- Média do valor do frete - 28 dias
    coalesce(avg(case when order_purchase_timestamp >= date('2018-07-01') - interval '28 days' then freight_value end), 0) as avg_frete_28d,

    -- Média do valor do frete - 56 dias
    coalesce(avg(case when order_purchase_timestamp >= date('2018-07-01') - interval '56 days' then freight_value end), 0) as avg_frete_56d,

    -- Média do valor do frete - 365 dias
    coalesce(avg(case when order_purchase_timestamp >= date('2018-07-01') - interval '365 days' then freight_value end), 0) as avg_frete_365d,

    -- SLA de entrega
    coalesce(avg(date_diff(order_estimated_delivery_date, order_approved_at)), 0) as avg_sla_entrega,

    -- Tempo médio de entrega do pedido
    coalesce(avg(date_diff(order_delivered_customer_date, order_approved_at)), 0) as avg_entrega,

    -- Tempo médio de transporte do pedido
    coalesce(avg(date_diff(order_delivered_customer_date, order_delivered_carrier_date)), 0) as avg_transp_ped,

    -- Número de pedidos atrasados
    coalesce(count(case when date_diff(order_estimated_delivery_date, order_delivered_customer_date)> 0 then 1 end), 0) as n_ped_atrasados,

    -- Percentual de pedidos atrasados - Vida
    coalesce(count(case when date_diff(order_estimated_delivery_date, order_delivered_customer_date) > 0 then 1 end) / count(*) * 100, 0) as pct_ped_atrasados_vida,

    -- Percentual de pedidos atrasados - 28 dias
    coalesce(count(case when order_purchase_timestamp >= date('2018-07-01') - interval '28 days' and date_diff(order_estimated_delivery_date, order_delivered_customer_date) > 0 then 1 end) / nullif(count(case when order_purchase_timestamp >= date('2018-07-01') - interval '28 days' then 1 end), 0) * 100, 0) as pct_ped_atrasados_28d,

    -- Percentual de pedidos atrasados - 56 dias
    coalesce(count(case when order_purchase_timestamp >= date('2018-07-01') - interval '56 days' and date_diff(order_estimated_delivery_date, order_delivered_customer_date) > 0 then 1 end) / nullif(count(case when order_purchase_timestamp >= date('2018-07-01') - interval '56 days' then 1 end), 0) * 100, 0) as pct_ped_atrasados_56d,

    -- Percentual de pedidos atrasados - 365 dias
    coalesce(count(case when order_purchase_timestamp >= date('2018-07-01') - interval '365 days' and date_diff(order_estimated_delivery_date, order_delivered_customer_date) > 0 then 1 end) / nullif(count(case when order_purchase_timestamp >= date('2018-07-01') - interval '365 days' then 1 end), 0) * 100, 0) as pct_ped_atrasados_365d,

    -- Tempo médio de entrega dos pedidos atrasados - Vida
    coalesce(avg(case when date_diff(order_estimated_delivery_date, order_delivered_customer_date) > 0 then date_diff(order_estimated_delivery_date, order_delivered_customer_date) end), 0) as avg_entrega_ped_atrasados_vida,

    -- Tempo médio de entrega dos pedidos atrasados - 28 dias
    coalesce(avg(case when order_purchase_timestamp >= date('2018-07-01') - interval '28 days' and date_diff(order_estimated_delivery_date, order_delivered_customer_date) > 0 then date_diff(order_estimated_delivery_date, order_delivered_customer_date) end), 0) as avg_entrega_ped_atrasados_28d,

    -- Tempo médio de entrega dos pedidos atrasados - 56 dias
    coalesce(avg(case when order_purchase_timestamp >= date('2018-07-01') - interval '56 days' and date_diff(order_estimated_delivery_date, order_delivered_customer_date) > 0 then date_diff(order_estimated_delivery_date, order_delivered_customer_date) end), 0) as avg_entrega_ped_atrasados_56d,

    -- Tempo médio de entrega dos pedidos atrasados - 365 dias
    coalesce(avg(case when order_purchase_timestamp >= date( '2018-07-01') - interval '365 days' and date_diff(order_estimated_delivery_date, order_delivered_customer_date) > 0 then date_diff(order_estimated_delivery_date, order_delivered_customer_date) end), 0) as avg_entrega_ped_atrasados_365d,

    -- Diferença média percentual(25%, 50%, 75%) entre a data de entrega estimada e data de entrega
    -- 25%
    percentile_cont(0.25) within group (order by date_diff(order_estimated_delivery_date, order_delivered_customer_date)) as diff_entrega_ped_25p,

    -- 50%
    percentile_cont(0.50) within group (order by date_diff(order_estimated_delivery_date, order_delivered_customer_date)) as diff_entrega_ped_50p,

    -- 75%
    percentile_cont(0.75) within group (order by date_diff(order_estimated_delivery_date, order_delivered_customer_date)) as diff_entrega_ped_75p,    

    -- Tempo Limite vendedor = shipping_limit_date - order_approved_at
    coalesce(avg(date_diff(shipping_limit_date, order_approved_at)), 0) as avg_tempo_limite_vendedor,

    -- Tempo Atraso vendedor = order_delivered_carrier_date - shipping_limit_date
    coalesce(avg(date_diff(order_delivered_carrier_date, shipping_limit_date)), 0) as avg_tempo_atraso_vendedor,

    -- Tempo transportadora = order_delivered_customer_date - order_delivered_carrier_date
    coalesce(avg(date_diff(order_delivered_customer_date, order_delivered_carrier_date)), 0) as avg_tempo_transportadora,

    -- Consistencia = (STDDEV(dias_despacho) / AVG (dias_despacho)
    coalesce(stddev(date_diff(order_estimated_delivery_date, order_delivered_customer_date)) / nullif(avg(date_diff(order_estimated_delivery_date, order_delivered_customer_date)),0), 0) as consistencia

FROM tb_pedidos
GROUP BY seller_id
ORDER BY seller_id